<a href="https://colab.research.google.com/github/deveki-gmail/sample-files/blob/main/FamilyTreeAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU langchain langchain-huggingface langchain-qdrant langchain-groq langchain-text-splitters
!pip install -q sentence-transformers qdrant-client

In [2]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [34]:
import requests
from langchain_core.documents import Document
response = requests.get("https://raw.githubusercontent.com/deveki-gmail/sample-files/refs/heads/main/DummyFamilyTree.txt")
response.raise_for_status()
docs = [Document(page_content=response.text, metadata={"source": "hand-entry"})]
print(docs)

[Document(metadata={'source': 'hand-entry'}, page_content='Family Lineage Breakdown\n• Generation 1 (Grandparents)\n\t• Arthur Pendelton (Born 1935) — Retired watchmaker and family patriarch.\n\t• Beatrice Vance (Born 1938) — Former botanical illustrator and avid gardener.\n• Generation 2 (Parents & Aunts/Uncles)\n\t• Edward Pendelton (Born 1962) — Arthur and Beatrice’s son; architect.\n\t• Margaret Hayes (Born 1964) — Edward’s spouse; high school literature teacher.\n\t• Clara Pendelton (Born 1965) — Arthur and Beatrice’s daughter; marine biologist.\n\t• Julian Vance (Born 1963) — Clara’s spouse; acoustic engineer (no relation to Beatrice despite the shared surname).\n• Generation 3 (Children & Cousins)\n\t• Leo Pendelton (Born 1990) — Son of Edward and Margaret; software developer.\n\t• Nora Pendelton (Born 1993) — Daughter of Edward and Margaret; graphic designer.\n\t• Sam Vance (Born 1991) — Son of Clara and Julian; chef.\n\t• Maya Vance (Born 1995) — Daughter of Clara and Julian; 

In [40]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=100, chunk_overlap=0
)
documents = text_splitter.split_documents(docs)
len(documents)


1

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

embedder = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID)
print(embedder)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [43]:
vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embedder,
    path="/tmp/my_lang_vs_2",
    collection_name="hr_docs_2",
)

RuntimeError: Storage folder /tmp/my_lang_vs_2 is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

In [44]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print(retriever)

tags=['QdrantVectorStore', 'HuggingFaceEmbeddings'] vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x7f9e6810d450> search_kwargs={'k': 5}


In [45]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. Cite section names. Say 'I don't know' if unsure."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

In [46]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [47]:
def format_docs(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        dl_meta  = doc.metadata.get("dl_meta", {})
        parts.append(f"[{i}]: \n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

In [48]:
def rag(query: str) -> str:
    docs         = retriever.invoke(query)
    print(docs)
    context      = format_docs(docs)
    prompt_value = RAG_PROMPT.invoke({"context": context, "question": query})
    response     = llm.invoke(prompt_value)
    return response.content

In [50]:
question = "Occupation of Leo Pendelton";
answer = rag(question)
print(answer)

[Document(metadata={'source': 'hand-entry', '_id': '417943e0-bd37-4c0d-b858-48912edef0fe', '_collection_name': 'hr_docs_2'}, page_content='Family Lineage Breakdown\n• Generation 1 (Grandparents)\n\t• Arthur Pendelton (Born 1935) — Retired watchmaker and family patriarch.\n\t• Beatrice Vance (Born 1938) — Former botanical illustrator and avid gardener.\n• Generation 2 (Parents & Aunts/Uncles)\n\t• Edward Pendelton (Born 1962) — Arthur and Beatrice’s son; architect.\n\t• Margaret Hayes (Born 1964) — Edward’s spouse; high school literature teacher.\n\t• Clara Pendelton (Born 1965) — Arthur and Beatrice’s daughter; marine biologist.\n\t• Julian Vance (Born 1963) — Clara’s spouse; acoustic engineer (no relation to Beatrice despite the shared surname).\n• Generation 3 (Children & Cousins)\n\t• Leo Pendelton (Born 1990) — Son of Edward and Margaret; software developer.\n\t• Nora Pendelton (Born 1993) — Daughter of Edward and Margaret; graphic designer.\n\t• Sam Vance (Born 1991) — Son of Clar